# Stage 4 v4 — ball inference on GPU (self-contained)

**One file. Set `CLIP` + `RANGE`, Run All.** Loads the trained `ball_model_v4.pt`
and a clip's `video.mp4` from Drive, runs the v4 TrackNet on GPU (batched), and
writes a real `ball.parquet` + `ball.meta.json` (`synthetic=false`) back to
Drive — a drop-in for the synthetic placeholder.

**Setup on Drive (`MyDrive/`):**
- `ball_model_v4.pt`  ← from training (already there)
- `pb_infer/<CLIP>/video.mp4`  ← upload the clip you want to analyze

**Runtime → Change runtime type → GPU.** CPU works but is ~11 s/frame.

Mirrors `stages/track_ball/track_ball_v4.py`; `postprocess()` is identical, so
the trajectory output matches the local module. Built by
`tools/build_infer_v4_nb.py`.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slow!)')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')


In [ ]:
try:
    import cv2
except Exception:
    !pip -q install opencv-python-headless
    import cv2
import numpy as np, pandas as pd


In [ ]:
# ===== KNOBS =====
CLIP    = 'pb_2min'          # folder under MyDrive/pb_infer/<CLIP>/ with video.mp4
WEIGHTS = DRIVE/'ball_model_v4.pt'
START   = 0                  # first frame to infer
MAXF    = None               # None = whole clip; or an int for a sub-range
OVERLAY = False              # write a debug overlay mp4 (only for short ranges!)
OVERLAY_MAXF = 600           # guard: refuse overlay if range longer than this

CONF_THRESH      = 0.30      # heatmap peak >= this -> a detection
MAX_GAP_FRAMES   = 8         # interpolate confirmed-detection gaps up to this
OUTLIER_MAX_STEP_PX = 250.0  # source px/frame; det this far from BOTH neighbours dropped
PROC_H, PROC_W   = 720, 1280
# frames per forward pass — scale to GPU memory. A 15GB T4 OOMs at 16 @ 720x1280
# (~0.9GB/sample), so keep it modest; big GPUs (A100) can push higher.
if torch.cuda.is_available():
    _gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 16 if _gb > 32 else (8 if _gb > 20 else 4)
else:
    BATCH = 1
SCHEMA_VERSION = 1

CLIP_DIR = DRIVE/'pb_infer'/CLIP
VIDEO = CLIP_DIR/'video.mp4'
assert VIDEO.exists(), f'missing {VIDEO} — upload the clip video to Drive'
assert Path(WEIGHTS).exists(), f'missing {WEIGHTS}'
print('clip', VIDEO, '| weights', WEIGHTS)


## TrackNet model (embedded verbatim from _tracknet_model.py)

In [ ]:
"""
TrackNetV2 PyTorch model — vendored from mareksubocz/TrackNet, adapted to
match the BatchNorm convention used in Andrew Dettor's pickleball-trained
SavedModel.

The TrackNet class below is reconstructed from the architecture described
in the TrackNetV2 paper (Sun et al., 2020) and the published source of
mareksubocz/TrackNet (https://github.com/mareksubocz/TrackNet, archived
Sep 2024). It is a 3-in-3-out heatmap-based small-object tracker:
input is 3 consecutive RGB frames stacked along channel axis (9 channels
total); output is 3 heatmaps, one per input frame, with values in [0, 1]
(after sigmoid) indicating ball-presence probability per pixel.

NOTE on BatchNorm:

Dettor's training code used Keras `BatchNormalization` layers with the
default `axis=-1` argument while feeding NCHW (channels-first) data.
With NCHW input, axis=-1 is the WIDTH axis, not the channels axis. So
his BN layers normalize over the rightmost spatial dimension instead
of channels. This is unconventional but it is the convention his
trained weights expect.

To stay faithful to those trained weights we cannot use the standard
`nn.BatchNorm2d` (which normalizes channels). Instead this file defines
`BatchNormOverWidth`: a custom 4-parameter normalization with parameter
size equal to the width dimension at the layer's position. The
TrackNet __init__ constructs each BN with the correct width parameter
for its position in the encoder / decoder.

The native input resolution is (288, 512). At each encoder downsample
the width halves (512 -> 256 -> 128 -> 64). In the decoder the width
doubles back up (64 -> 128 -> 256 -> 512). Each BatchNormOverWidth
instance has parameter size equal to the width at its position.

If TrackNet is ever used at a non-(288, 512) resolution, every BN will
need to be re-instantiated with the new widths. The model's __init__
takes an optional `input_shape` argument to support this — defaults to
(288, 512) which is what Dettor's weights were trained on.

------------------------------------------------------------------------
Original mareksubocz/TrackNet license:

MIT License

Copyright (c) 2023 Marek Subocz

Permission is hereby granted, free of charge, to any person obtaining
a copy of this software and associated documentation files (the
"Software"), to deal in the Software without restriction, including
without limitation the rights to use, copy, modify, merge, publish,
distribute, sublicense, and/or sell copies of the Software, and to
permit persons to whom the Software is furnished to do so, subject to
the following conditions:

The above copyright notice and this permission notice shall be included
in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS
OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
------------------------------------------------------------------------
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class BatchNormOverWidth(nn.Module):
    """Per-width-position batch normalization.

    For input (N, C, H, W), normalizes over (N, C, H) — i.e. computes
    one mean/var per W position. Has 4 learnable/buffered parameters
    each of size W: gamma, beta, running_mean, running_var.

    This matches the behavior of Keras `BatchNormalization(axis=-1)`
    applied to channels-first data, which is what Dettor's training
    pipeline did.

    Forward pass uses running statistics (eval mode); we never train
    this model in our pipeline. Train-mode behavior is implemented for
    completeness but not exercised.
    """

    def __init__(self, num_features: int, eps: float = 1e-3,
                 momentum: float = 0.99):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        # Learnable affine parameters
        self.weight = nn.Parameter(torch.ones(num_features))   # gamma
        self.bias = nn.Parameter(torch.zeros(num_features))    # beta
        # Running statistics (buffers, not parameters)
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))
        self.register_buffer("num_batches_tracked",
                             torch.tensor(0, dtype=torch.long))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, C, H, W)
        if x.shape[-1] != self.num_features:
            raise RuntimeError(
                f"BatchNormOverWidth: input width {x.shape[-1]} does not "
                f"match num_features {self.num_features}. This BN was "
                f"constructed for a different position in the model."
            )
        if self.training:
            # Compute mean/var over (N, C, H) for each W position
            mean = x.mean(dim=(0, 1, 2))
            var = x.var(dim=(0, 1, 2), unbiased=False)
            # Update running stats (this branch is not used in our pipeline)
            with torch.no_grad():
                self.running_mean.mul_(self.momentum).add_(
                    mean.detach(), alpha=1.0 - self.momentum)
                self.running_var.mul_(self.momentum).add_(
                    var.detach(), alpha=1.0 - self.momentum)
                self.num_batches_tracked.add_(1)
        else:
            mean = self.running_mean
            var = self.running_var
        # Apply: (x - mean) / sqrt(var + eps) * gamma + beta
        # Broadcast: mean/var/weight/bias are (W,), x is (N, C, H, W)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return x_norm * self.weight + self.bias


class TrackNet(nn.Module):
    """TrackNetV2: VGG16-encoder + decoder with skip connections, 3-in-3-out.

    Args:
        in_channels: number of input channels. Default 9 (3 frames * 3 RGB).
        out_channels: number of output heatmap channels. Default 3.
        dropout_rate: dropout applied inside each conv sublayer. Default 0.0.
        input_shape: (H, W) of the model's expected input. Default (288, 512).
            Used to construct BatchNormOverWidth instances with correct
            per-layer widths. If you change this, you must retrain — the
            BN parameter sizes change with width.

    Input shape: (batch, in_channels, H, W). Default H=288, W=512.
    Output shape: (batch, out_channels, H, W), values in [0, 1] after sigmoid.
    """

    def __init__(self, in_channels: int = 9, out_channels: int = 3,
                 dropout_rate: float = 0.0,
                 input_shape: tuple = (288, 512)):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.input_h, self.input_w = input_shape

        # Compute width at each level of the encoder (after each maxpool)
        # and decoder (after each upsample). Pool reduces by 2; upsample
        # increases by 2.
        w0 = self.input_w           # 512  (full res, encoder block 1)
        w1 = w0 // 2                # 256  (after pool 1, encoder block 2)
        w2 = w1 // 2                # 128  (after pool 2, encoder block 3)
        w3 = w2 // 2                # 64   (after pool 3, encoder block 4 / bottleneck)
        # Decoder mirrors the encoder
        wd3 = w2                    # 128  (after upsample 1, decoder block 3)
        wd2 = w1                    # 256  (after upsample 2, decoder block 2)
        wd1 = w0                    # 512  (after upsample 3, decoder block 1)

        # ---- Encoder ----
        self.enc1 = self._make_conv_block(in_channels, 64, num=2,
                                          width=w0,
                                          dropout_rate=dropout_rate)
        self.enc2 = self._make_conv_block(64, 128, num=2,
                                          width=w1,
                                          dropout_rate=dropout_rate)
        self.enc3 = self._make_conv_block(128, 256, num=3,
                                          width=w2,
                                          dropout_rate=dropout_rate)
        self.enc4 = self._make_conv_block(256, 512, num=3,
                                          width=w3,
                                          dropout_rate=dropout_rate)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # ---- Decoder ----
        self.dec3 = self._make_conv_block(768, 256, num=3,
                                          width=wd3,
                                          dropout_rate=dropout_rate)
        self.dec2 = self._make_conv_block(384, 128, num=2,
                                          width=wd2,
                                          dropout_rate=dropout_rate)
        self.dec1 = self._make_conv_block(192, 64, num=2,
                                          width=wd1,
                                          dropout_rate=dropout_rate)

        # Final 1x1 conv (no BN follows it)
        self.head = nn.Conv2d(64, out_channels, kernel_size=(1, 1), padding=0)

    @staticmethod
    def _make_conv_sublayer(in_channels: int, out_channels: int,
                             width: int,
                             dropout_rate: float = 0.0) -> nn.Sequential:
        """Conv2d(3x3, same padding) -> ReLU -> BatchNormOverWidth -> [Dropout].

        Note: BatchNormOverWidth's parameter size is `width`, NOT
        `out_channels`, because Dettor's training applied BN over the
        width axis (Keras default axis=-1 with NCHW data).
        """
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=(3, 3),
                      padding="same"),
            nn.ReLU(inplace=True),
            BatchNormOverWidth(num_features=width),
        ]
        if dropout_rate > 1e-15:
            layers.append(nn.Dropout(dropout_rate))
        return nn.Sequential(*layers)

    def _make_conv_block(self, in_channels: int, out_channels: int,
                         num: int, width: int,
                         dropout_rate: float = 0.0) -> nn.Sequential:
        """Stack of `num` conv sublayers, each followed by BN-over-width."""
        sublayers = [self._make_conv_sublayer(in_channels, out_channels,
                                               width=width,
                                               dropout_rate=dropout_rate)]
        for _ in range(num - 1):
            sublayers.append(self._make_conv_sublayer(
                out_channels, out_channels,
                width=width, dropout_rate=dropout_rate))
        return nn.Sequential(*sublayers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        d3 = F.interpolate(e4, scale_factor=2, mode="nearest")
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = F.interpolate(d3, scale_factor=2, mode="nearest")
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = F.interpolate(d2, scale_factor=2, mode="nearest")
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        out = self.head(d1)
        out = torch.sigmoid(out)
        return out


def count_layers(model: nn.Module) -> dict:
    """Diagnostic helper: count Conv2d and BatchNormOverWidth layers."""
    n_conv = sum(1 for m in model.modules() if isinstance(m, nn.Conv2d))
    n_bn = sum(1 for m in model.modules() if isinstance(m, BatchNormOverWidth))
    return {"conv": n_conv, "bn": n_bn}


## Trajectory post-processing (verbatim from track_ball_v4.py)

In [ ]:
def postprocess(dets: dict, frames: list) -> list:
    """dets: frame -> (x, y, conf) for frames whose peak >= CONF. frames: full
    ordered frame list to emit rows for. Returns per-frame row dicts."""
    conf_frames = sorted(dets.keys())
    # 1) drop isolated velocity outliers (far from BOTH neighbors)
    kept = set(conf_frames)
    for i, f in enumerate(conf_frames):
        x, y, _ = dets[f]
        bad = []
        for j in (i - 1, i + 1):
            if 0 <= j < len(conf_frames):
                g = conf_frames[j]
                px, py, _ = dets[g]
                if np.hypot(x - px, y - py) > OUTLIER_MAX_STEP_PX * max(1, abs(f - g)):
                    bad.append(True)
                else:
                    bad.append(False)
        if bad and all(bad):  # impossible jump from every neighbor present
            kept.discard(f)
    conf = [f for f in conf_frames if f in kept]
    confset = set(conf)

    # 2) interpolate short gaps between consecutive confirmed detections
    interp = {}
    for a, b in zip(conf, conf[1:]):
        gap = b - a
        if 1 < gap <= MAX_GAP_FRAMES:
            xa, ya, _ = dets[a]
            xb, yb, _ = dets[b]
            for k in range(1, gap):
                t = k / gap
                interp[a + k] = (xa + t * (xb - xa), ya + t * (yb - ya))

    rows = []
    for f in frames:
        if f in confset:
            x, y, c = dets[f]
            rows.append({"frame_idx": f, "pixel_x": float(x), "pixel_y": float(y),
                         "visible": True, "confidence": float(c), "interpolated": False})
        elif f in interp:
            x, y = interp[f]
            rows.append({"frame_idx": f, "pixel_x": float(x), "pixel_y": float(y),
                         "visible": False, "confidence": np.nan, "interpolated": True})
        else:
            rows.append({"frame_idx": f, "pixel_x": np.nan, "pixel_y": np.nan,
                         "visible": False, "confidence": np.nan, "interpolated": False})
    return rows


## Run inference (batched on GPU)

In [ ]:
import datetime as dt, json, time
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load(str(WEIGHTS), map_location=dev)
ishape = tuple(ck.get('input_shape', (PROC_H, PROC_W)))
model = TrackNet(in_channels=ck.get('in_channels', 9),
                 out_channels=ck.get('out_channels', 1), input_shape=ishape).to(dev)
model.load_state_dict(ck['state_dict']); model.eval()
print('model @', ishape, 'on', dev)

cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), VIDEO
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS)
sw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); sh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
sx, sy = sw / PROC_W, sh / PROC_H
start = max(0, START); end = n_total if MAXF is None else min(n_total, start + MAXF)
frames = list(range(start, end))
print(f'video {sw}x{sh}@{fps:.1f}, {n_total} frames; inferring [{start},{end})  batch {BATCH}')

want_overlay = OVERLAY and (end - start) <= OVERLAY_MAXF
if OVERLAY and not want_overlay:
    print(f'OVERLAY skipped: range {end-start} > OVERLAY_MAXF {OVERLAY_MAXF}')
src_cache = {} if want_overlay else None

def to_proc(fr):
    rgb = cv2.cvtColor(cv2.resize(fr, (PROC_W, PROC_H), interpolation=cv2.INTER_AREA),
                       cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return rgb.transpose(2, 0, 1)

@torch.no_grad()
def flush(batch, dets):
    if not batch:
        return
    t = torch.from_numpy(np.stack([s for _, s in batch])).to(dev)
    with torch.amp.autocast('cuda', enabled=dev == 'cuda'):
        hm = model(t)[:, 0]
    hm = hm.float().cpu().numpy()
    for k, (cf, _) in enumerate(batch):
        h = hm[k]; iy, ix = np.unravel_index(int(h.argmax()), h.shape)
        c = float(h[iy, ix])
        if c >= CONF_THRESH:
            dets[cf] = (ix * sx, iy * sy, c)

cap.set(cv2.CAP_PROP_POS_FRAMES, start)
buf, batch, dets = [], [], {}
fidx = start; t0 = time.time()
while fidx < end:
    ok, fr = cap.read()
    if not ok:
        break
    if src_cache is not None:
        src_cache[fidx] = fr
    buf.append(to_proc(fr))
    if len(buf) > 3:
        buf.pop(0)
    if len(buf) == 3:
        batch.append((fidx - 1, np.concatenate(buf, axis=0)))
        if len(batch) >= BATCH:
            flush(batch, dets); batch = []
    fidx += 1
    if (fidx - start) % 500 == 0:
        el = time.time() - t0
        print(f'  {fidx-start}/{len(frames)}  ({(fidx-start)/max(el,1e-9):.1f} fps)')
flush(batch, dets)
cap.release()
print(f'detections: {len(dets)} / {len(frames)} frames in {time.time()-t0:.1f}s')


## Write ball.parquet + meta (+ optional overlay), back to Drive

In [ ]:
rows = postprocess(dets, frames)
df = pd.DataFrame(rows)
df.insert(0, 'schema_version', SCHEMA_VERSION)
df['visible'] = df['visible'].astype(bool)
df['interpolated'] = df['interpolated'].astype(bool)
df['confidence'] = df['confidence'].astype('float32')
out_parquet = CLIP_DIR/'ball.parquet'; out_meta = CLIP_DIR/'ball.meta.json'
df.to_parquet(out_parquet, index=False)

n_vis = int(df['visible'].sum()); n_interp = int(df['interpolated'].sum()); n = len(df)
meta = {'schema_version': SCHEMA_VERSION, 'synthetic': False, 'video_path': str(VIDEO),
        'video_frame_count': n_total, 'video_fps': float(fps),
        'video_width': sw, 'video_height': sh,
        'detector': {'tool': 'stages/track_ball/infer_v4.ipynb', 'weights': str(WEIGHTS),
                     'proc_hw': [PROC_H, PROC_W], 'conf_thresh': CONF_THRESH,
                     'max_gap_frames': MAX_GAP_FRAMES, 'batch': BATCH},
        'range': [start, end],
        'stats': {'frames': n, 'frames_visible': n_vis, 'frames_interpolated': n_interp,
                  'frames_not_visible': n - n_vis - n_interp,
                  'visible_frac': round(n_vis / max(n, 1), 4),
                  'detect_frac': round((n_vis + n_interp) / max(n, 1), 4)},
        'completed_at_utc': dt.datetime.now(dt.timezone.utc).isoformat()}
Path(out_meta).write_text(json.dumps(meta, indent=1) + '\n', encoding='utf-8')
print(f'wrote {out_parquet}')
print(f'  {n_vis} visible, {n_interp} interp, {n-n_vis-n_interp} not-visible '
      f"(detect_frac {meta['stats']['detect_frac']})")

if src_cache is not None:
    op = CLIP_DIR/'_ball_check.mp4'
    rmap = {int(r.frame_idx): r for r in df.itertuples(index=False)}
    w = cv2.VideoWriter(str(op), cv2.VideoWriter_fourcc(*'mp4v'), fps, (sw, sh))
    for f in sorted(src_cache):
        fr = src_cache[f]; r = rmap.get(f)
        if r is not None and not (isinstance(r.pixel_x, float) and np.isnan(r.pixel_x)):
            col = (0, 255, 0) if r.visible else (0, 255, 255)
            cv2.circle(fr, (int(r.pixel_x), int(r.pixel_y)), 12, col, 3, cv2.LINE_AA)
        w.write(fr)
    w.release(); print('wrote overlay', op)


## Done
- Download `MyDrive/pb_infer/<CLIP>/ball.parquet` + `ball.meta.json` → the clip
  folder under `data/` locally. It is a drop-in for the synthetic ball.
- `detect_frac` is the share of frames with a ball (visible + interpolated).
  Active rallies should be high; dead time between points is legitimately low.
- If `OVERLAY=True` on a short range, eyeball `_ball_check.mp4`: green = detected,
  yellow = interpolated. Dots should sit on the ball.
